# 45. Aliphatic_long_chain 완전 해결 (제출 이후 개선)

## 목적
insert_atom_multi_chain을 넓어진 SMARTS(탄소+에테르산소 혼합 사슬)에
맞춰 재작업하여, 1회 삽입(insert_atom)만으로는 근본 해결이 안 되던
긴 사슬 케이스를 완전 해결까지 도달시킨다.

## 배경 (제출 완료 시점, 42개 규칙 기준 최종 수치)
- 커버리지 50.2%, 완전해결 52%, 부분개선 80% (valid set)
- 커버리지 49.9%, 완전해결 50%, 부분개선 72% (test set, 1회 검증 완료)
- 3-에이전트 배치검증(100개, 캐시버그 수정 후): 실패19/재검토35/
  사람검토45/승인1
- 알려진 문제: Aliphatic_long_chain이 stuck의 다수(32~34/48건) 차지.
  SMARTS는 안전하게 확장했으나(과산화물 방지 포함), insert_atom이
  1회 1지점만 삽입해 완전 해결까지 못 감(1단계 진행 후 재진단에서
  다시 걸림)
- 원인 진단 완료: insert_atom_multi_chain의 사슬추적/삽입로직을
  새 SMARTS(탄소+에테르산소)에 맞게 재작업 필요
- 이제 제출 마감 압박 없음 - 신중하게 검증하며 진행 가능
- test set은 이미 1회 사용 완료, 앞으로는 절대 재사용 금지
  (개발 중에는 valid set만 사용)

In [1]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 600, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 600 (delta 26), reused 39 (delta 12), pack-reused 540 (from 1)
Receiving objects: 100% (600/600), 5.50 MiB | 20.47 MiB/s, done.
Resolving deltas: 100% (343/343), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib, random, json
import numpy as np
from rdkit import Chem
from rdkit.Chem import FilterCatalog

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory

data = load_tox21_clean(random_state=7)
n_rules = len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])
print(f"라이브러리 규칙 수: {n_rules}")
clear_failure_memory()

[06:44:45] WARNING: not removing hydrogen atom without neighbors
[06:44:46] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:44:46] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:44:46] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:44:46] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:44:47] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:44:47] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:44:47] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:44:48] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:44:48] WARNING: not removing hydrogen atom without neighbors


라이브러리 규칙 수: 42


In [5]:
count_baseline = 0
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    if any(get_replacement_candidates(x['rule_name']) is not None for x in p):
        count_baseline += 1
print(f"현재 커버리지: {count_baseline}/{len(data['smiles_valid'])} ({count_baseline/len(data['smiles_valid'])*100:.1f}%)")

# Aliphatic_long_chain 단독 stuck 케이스 재현
single_problem_molecules = []
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    known = [x for x in p if get_replacement_candidates(x['rule_name']) is not None]
    if len(known) == 1 and known[0]['rule_name'] == 'Aliphatic_long_chain':
        single_problem_molecules.append(s)

print(f"Aliphatic_long_chain 단독 문제 분자: {len(single_problem_molecules)}개")

stuck_count = 0
for s in single_problem_molecules[:20]:
    result = iterative_fix_loop(s, max_iterations=10)
    if result['status'] != 'success':
        stuck_count += 1

print(f"20개 표본 중 미해결: {stuck_count}건 (개선 전 baseline)")

현재 커버리지: 589/1173 (50.2%)
Aliphatic_long_chain 단독 문제 분자: 95개
20개 표본 중 미해결: 18건 (개선 전 baseline)


elif edit_type == "insert_atom_multi_chain":
        # 긴 지방족 사슬 전용(탄소 또는 비카르보닐 에테르 산소로 구성된
        # 사슬 모두 인식): 매치 시작점에서 양쪽 방향을 모두 추적해 더 긴
        # 쪽을 진짜 사슬로 채택한 뒤, 필요한 만큼 산소를 균등 삽입
        start_idx = match[candidate["chain_start_idx_in_pattern"]]

        def _is_chain_member(atom):
            if atom.GetSymbol() == 'C' and not atom.GetIsAromatic():
                return True
            if atom.GetSymbol() == 'O' and atom.GetDegree() == 2 and not atom.GetIsAromatic():
                for nb in atom.GetNeighbors():
                    for bond in nb.GetBonds():
                        if bond.GetBondTypeAsDouble() == 2.0 and nb.GetSymbol() == 'C':
                            other = bond.GetOtherAtom(nb)
                            if other.GetSymbol() == 'O':
                                return False
                return True
            return False

        def _trace_chain(mol, start, avoid):
            chain = [start]
            current = start
            prev = avoid

In [12]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_multi_bond":
        pairs = candidate.get("target_pairs_in_pattern", info.get("target_pairs_in_pattern"))
        ring_atoms_pattern = candidate.get("ring_atoms_in_pattern", info.get("ring_atoms_in_pattern"))
        ring_bonds_pattern = candidate.get("ring_bonds_in_pattern", info.get("ring_bonds_in_pattern"))

        for pair in pairs:
            idx_c = match[pair[0]]
            idx_o = match[pair[1]]
            bond = rwmol.GetBondBetweenAtoms(idx_c, idx_o)
            if bond is None:
                return None
            bond.SetBondType(Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
            rwmol.GetAtomWithIdx(idx_c).SetNoImplicit(False)

        ring_indices = [match[i] for i in ring_atoms_pattern]
        for a in ring_indices:
            rwmol.GetAtomWithIdx(a).SetIsAromatic(True)

        for b1, b2 in ring_bonds_pattern:
            bidx1, bidx2 = match[b1], match[b2]
            rbond = rwmol.GetBondBetweenAtoms(bidx1, bidx2)
            if rbond is None:
                continue
            rbond.SetBondType(Chem.BondType.AROMATIC)
            rbond.SetIsAromatic(True)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "remove_atom":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust4(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust4(center_idx, to_remove)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(center_new).SetFormalCharge(0)

    elif edit_type == "cleave_bond":
        pair = candidate["cleave_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)
        for idx in (idx1, idx2):
            rwmol.GetAtomWithIdx(idx).SetNoImplicit(False)

    elif edit_type == "open_epoxide":
        pair = candidate["break_pair_in_pattern"]
        idx_o = match[pair[0]]
        idx_c_break = match[pair[1]]

        bond = rwmol.GetBondBetweenAtoms(idx_o, idx_c_break)
        if bond is None:
            return None
        rwmol.RemoveBond(idx_o, idx_c_break)

        frag = Chem.MolFromSmiles("O")
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(idx_c_break, offset, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx_c_break).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(offset).SetNoImplicit(False)

    elif edit_type == "insert_atom":
        pair = candidate["insert_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]

        if rwmol.GetAtomWithIdx(idx1).GetSymbol() == 'O' or rwmol.GetAtomWithIdx(idx2).GetSymbol() == 'O':
            return None

        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)

        new_atom = Chem.Atom(candidate["param"])
        new_idx = rwmol.AddAtom(new_atom)
        rwmol.AddBond(idx1, new_idx, Chem.BondType.SINGLE)
        rwmol.AddBond(new_idx, idx2, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx1).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx2).SetNoImplicit(False)

    elif edit_type == "insert_atom_multi_chain":
        # 긴 지방족 사슬 전용: start_idx에서 양방향을 각각 추적한 뒤
        # 병합해 실제 사슬 전체를 잡는다 (기존: 한쪽만 채택하는 버그
        # -> start_idx가 사슬 중간이면 절반만 인식되어 stuck 유발).
        start_idx = match[candidate["chain_start_idx_in_pattern"]]

        def _is_chain_member(atom):
            if atom.GetSymbol() == 'C' and not atom.GetIsAromatic():
                return True
            if atom.GetSymbol() == 'O' and atom.GetDegree() == 2 and not atom.GetIsAromatic():
                for nb in atom.GetNeighbors():
                    for bond in nb.GetBonds():
                        if bond.GetBondTypeAsDouble() == 2.0 and nb.GetSymbol() == 'C':
                            other = bond.GetOtherAtom(nb)
                            if other.GetSymbol() == 'O':
                                return False
                return True
            return False

        def _trace_chain(mol, start, avoid, visited):
            chain = [start]
            local_visited = set(visited)
            local_visited.add(start)
            current = start
            prev = avoid
            while True:
                atom_cur = mol.GetAtomWithIdx(current)
                if not _is_chain_member(atom_cur):
                    break
                next_candidates = [n.GetIdx() for n in atom_cur.GetNeighbors()
                                    if n.GetIdx() != prev and n.GetIdx() not in local_visited
                                    and _is_chain_member(n)]
                if not next_candidates:
                    break
                best_sub = []
                for cand in next_candidates:
                    sub = _trace_chain(mol, cand, current, local_visited)
                    if len(sub) > len(best_sub):
                        best_sub = sub
                prev, current = current, next_candidates[0]
                chain.extend(best_sub)
                break
            return chain

        start_atom = mol.GetAtomWithIdx(start_idx)
        neighbor_options = [n.GetIdx() for n in start_atom.GetNeighbors() if _is_chain_member(n)]

        if len(neighbor_options) >= 2:
            traces = [_trace_chain(mol, nb, start_idx, {start_idx}) for nb in neighbor_options]
            traces.sort(key=len, reverse=True)
            chain_atoms = list(reversed(traces[0])) + [start_idx] + traces[1]
        elif len(neighbor_options) == 1:
            chain_atoms = [start_idx] + _trace_chain(mol, neighbor_options[0], start_idx, {start_idx})
        else:
            chain_atoms = [start_idx]

        if len(chain_atoms) < 4:
            return None

        n = len(chain_atoms)
        num_inserts = max(1, (n - 1) // 3)
        step = n / (num_inserts + 1)
        insert_positions = sorted(set(int(round(step * (i + 1))) for i in range(num_inserts)))
        insert_positions = [p for p in insert_positions if 0 < p < n]

        insert_after = [chain_atoms[p - 1] for p in insert_positions]
        if not insert_after:
            return None

        added = 0
        for a_idx in insert_after:
            a_pos = chain_atoms.index(a_idx)
            b_idx = chain_atoms[a_pos + 1]
            if mol.GetAtomWithIdx(a_idx).GetSymbol() == 'O' or mol.GetAtomWithIdx(b_idx).GetSymbol() == 'O':
                continue
            bond = rwmol.GetBondBetweenAtoms(a_idx, b_idx)
            if bond is None:
                continue
            rwmol.RemoveBond(a_idx, b_idx)
            new_o = rwmol.AddAtom(Chem.Atom(8))
            rwmol.AddBond(a_idx, new_o, Chem.BondType.SINGLE)
            rwmol.AddBond(new_o, b_idx, Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(a_idx).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(b_idx).SetNoImplicit(False)
            added += 1

        if added == 0:
            return None

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        ring_set = set(ring_indices)
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            for n in ratom.GetNeighbors():
                nidx = n.GetIdx()
                if nidx not in ring_set and nidx not in (anchor_idx1, anchor_idx2):
                    return None

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        allow_counterion = candidate.get("allow_counterion", False)
        allow_aromatic_zero_h = candidate.get("allow_aromatic_zero_h", False)

        if edit_type != "cleave_bond" and not allow_counterion and '.' in new_smiles:
            is_valid = False
        for atom in check_mol.GetAtoms():
            if allow_aromatic_zero_h and atom.GetIsAromatic() and atom.GetSymbol() == 'N':
                continue
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [13]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter

single_problem_check = []
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    known = [x for x in p if get_replacement_candidates(x['rule_name']) is not None]
    if len(known) == 1 and known[0]['rule_name'] == 'Aliphatic_long_chain':
        single_problem_check.append(s)

print(f"단독 문제 분자 수: {len(single_problem_check)} (95 근처여야 정상)")

sample = single_problem_check[:20]

status_counter = Counter()
results = []

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("\n--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck인 케이스 ---")
for smi, status, loop_result in results:
    if status == 'stuck':
        print(f"원본: {smi}")
        print(f"결과: {loop_result}\n")

단독 문제 분자 수: 95 (95 근처여야 정상)

--- 상태 분포 (20개 표본) ---
stuck: 18
success: 2

--- 여전히 stuck인 케이스 ---
원본: CCCCCCCCCCCCCCCCCCOCC(O)CO
결과: {'status': 'stuck', 'reason': "시도한 규칙 ['Aliphatic_long_chain'] 모두 치환 실패", 'reason_detail': "이 단계에서 known 규칙 ['Aliphatic_long_chain'] 전부를 순서대로 시도했으나 모두 실행에 실패했습니다(memory-skip 표시는 이전에 실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.", 'final_smiles': 'CCOCCCCCCCCCCCCCCCCOCC(O)CO', 'history': [{'step': 0, 'smiles': 'CCCCCCCCCCCCCCCCCCOCC(O)CO', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [1, 2, 3, 4]}]}, {'step': 1, 'smiles': 'CCOCCCCCCCCCCCCCCCCOCC(O)CO', 'fixed_rule': 'Aliphatic_long_chain', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'ether-inserted chain (O in middle)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [1, 2, 3, 4]}]}], 'skipped_rules': [], 'skipped_details': []}

원본: CCCCCCCCCCCCSC#N
결과: {'status': 'stuck', '

In [14]:
info = get_replacement_candidates('Aliphatic_long_chain')
for i, c in enumerate(info['candidates']):
    print(f"idx={i} | name={c['name']!r} | edit_type={c['edit_type']}")

idx=0 | name='ether-inserted chain (O in middle)' | edit_type=insert_atom
idx=1 | name='multi-ether chain (multiple O inserted for long chains)' | edit_type=insert_atom_multi_chain


In [15]:
info = get_replacement_candidates('Aliphatic_long_chain')
pattern = Chem.MolFromSmarts(info["problem_smarts"])

stuck_smiles_batch2 = [
    "CCCCCCCCCCCCCCCCCCOCC(O)CO",
    "CCCCCCCCCCCCSC#N",
    "CCOCCOCCO",
    "CCCCCCC(O)CO",
    "CCCCCCCCCCCCSC",
    "CCCCCCCCNC",
    "CCCCCCCCCCCCCCCCC(=O)O",
    "CCCCCCCCOCCC#N",
    "COCCOCCOCCOC",
    "CCCCCCCCCOC(C)=O",
]

for smi in stuck_smiles_batch2:
    result = apply_atom_edit_from_rule(smi, 'Aliphatic_long_chain', candidate_idx=1)
    if result is None:
        print(f"{smi} -> None (실행 실패)\n")
        continue
    new_mol = Chem.MolFromSmiles(result['new_smiles'])
    still_matches = new_mol.HasSubstructMatch(pattern) if new_mol else None
    print(f"원본: {smi}")
    print(f"결과: {result['new_smiles']}  (valid={result['is_valid']}, 여전히 매치={still_matches})\n")

원본: CCCCCCCCCCCCCCCCCCOCC(O)CO
결과: CCCOCCCOCCOCCCOCCCOCCOCCOCC(O)CO  (valid=True, 여전히 매치=True)

원본: CCCCCCCCCCCCSC#N
결과: CCCOCCCOCCCOCCCSC#N  (valid=True, 여전히 매치=True)

CCOCCOCCO -> None (실행 실패)

원본: CCCCCCC(O)CO
결과: CCCOCCOCC(O)CO  (valid=True, 여전히 매치=True)

원본: CCCCCCCCCCCCSC
결과: CCCOCCCOCCCOCCCSC  (valid=True, 여전히 매치=True)

원본: CCCCCCCCNC
결과: CCCOCCOCCCNC  (valid=True, 여전히 매치=True)

원본: CCCCCCCCCCCCCCCCC(=O)O
결과: CCCOCCCOCCOCCCOCCCOCCC(=O)O  (valid=True, 여전히 매치=True)

원본: CCCCCCCCOCCC#N
결과: CCCOCCCOCCOCCC#N  (valid=True, 여전히 매치=True)

원본: COCCOCCOCCOC
결과: COCOCOCOCOCOCOC  (valid=True, 여전히 매치=True)

원본: CCCCCCCCCOC(C)=O
결과: CCCOCCCOCCCOC(C)=O  (valid=True, 여전히 매치=True)



In [16]:
!cat src/tools/molecule_editor.py


from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib

def _library_version_hash():
    """현재 REPLACEMENT_LIBRARY 내용의 해시값. 라이브러리가 바뀌면
    자동으로 다른 값이 나와서, 캐시 키에 포함시키면 라이브러리 변경 시
    이전 캐시가 자동으로 무효화된다(수동 clear_failure_memory() 호출 불필요)."""
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    """규칙 라이브러리가 업데이트된 뒤(reload 후) 호출해 캐시를 초기화."""
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = 

In [17]:
from src.tools.toxicophore_detector import detect_toxicophores

stuck_smiles_batch2 = [
    "CCCCCCCCCCCCCCCCCCOCC(O)CO",
    "CCCCCCCCCCCCSC#N",
    "CCOCCOCCO",
    "CCCCCCC(O)CO",
    "CCCCCCCCCCCCSC",
    "CCCCCCCCNC",
    "CCCCCCCCCCCCCCCCC(=O)O",
    "CCCCCCCCOCCC#N",
    "COCCOCCOCCOC",
    "CCCCCCCCCOC(C)=O",
]

for smi in stuck_smiles_batch2:
    result = apply_atom_edit_from_rule(smi, 'Aliphatic_long_chain', candidate_idx=1)
    if result is None:
        print(f"{smi} -> None (실행 실패)\n")
        continue
    new_smi = result['new_smiles']
    problems = detect_toxicophores(new_smi)
    still_flagged = any(p['rule_name'] == 'Aliphatic_long_chain' for p in problems)
    print(f"원본: {smi}")
    print(f"결과: {new_smi}")
    print(f"진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): {still_flagged}\n")

원본: CCCCCCCCCCCCCCCCCCOCC(O)CO
결과: CCCOCCCOCCOCCCOCCCOCCOCCOCC(O)CO
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

원본: CCCCCCCCCCCCSC#N
결과: CCCOCCCOCCCOCCCSC#N
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

CCOCCOCCO -> None (실행 실패)

원본: CCCCCCC(O)CO
결과: CCCOCCOCC(O)CO
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

원본: CCCCCCCCCCCCSC
결과: CCCOCCCOCCCOCCCSC
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

원본: CCCCCCCCNC
결과: CCCOCCOCCCNC
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

원본: CCCCCCCCCCCCCCCCC(=O)O
결과: CCCOCCCOCCOCCCOCCCOCCC(=O)O
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

원본: CCCCCCCCOCCC#N
결과: CCCOCCCOCCOCCC#N
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

원본: COCCOCCOCCOC
결과: COCOCOCOCOCOCOC
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True

원본: CCCCCCCCCOC(C)=O
결과: CCCOCCCOCCCOC(C)=O
진짜 재진단 결과 (Aliphatic_long_chain 여전히 있음?): True



In [18]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib

def _library_version_hash():
    """현재 REPLACEMENT_LIBRARY 내용의 해시값. 라이브러리가 바뀌면
    자동으로 다른 값이 나와서, 캐시 키에 포함시키면 라이브러리 변경 시
    이전 캐시가 자동으로 무효화된다(수동 clear_failure_memory() 호출 불필요)."""
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    """규칙 라이브러리가 업데이트된 뒤(reload 후) 호출해 캐시를 초기화."""
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]
            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue
            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue
            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')
            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def _candidate_order_for_rule(rule_name: str, preferred_idx: int):
    """해당 규칙의 전체 candidate 인덱스를, preferred_idx를 우선으로 하고
    나머지는 순서대로 뒤에 붙여서 반환. 이전 버전은 preferred_idx 하나만
    시도하고 실패하면 바로 그 규칙 전체를 포기했음 -> 같은 규칙 안에
    더 나은 candidate(예: insert_atom_multi_chain)가 있어도 절대
    시도되지 않는 버그가 있었음."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return [preferred_idx]
    n = len(info['candidates'])
    order = [preferred_idx] if 0 <= preferred_idx < n else []
    order += [i for i in range(n) if i != preferred_idx]
    return order


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini",
                        use_failure_memory: bool = True):
    """진단->치환->재평가를 반복.
    use_failure_memory=True(기본): 세션 전체에 걸쳐 "이 분자 상태 + 이
    규칙 + 이 candidate_idx" 조합이 이미 실패한 적 있으면 재시도하지
    않고 즉시 건너뜀(propose_fix 재호출 없이 스킵). 규칙 라이브러리를
    수정한 뒤에는 clear_failure_memory()를 호출해 캐시를 초기화해야 함."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            problem_reason = "규칙 기반(리스트 순서대로)"
            ordered_rules = [p['rule_name'] for p in known_problems]

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                preferred_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if preferred_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                preferred_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스 우선, 실패 시 같은 규칙 내 다른 candidate로 재시도)"

            # 같은 규칙 안에서도 candidate를 순서대로 재시도 (핵심 수정)
            rule_fixed = None
            for try_idx in _candidate_order_for_rule(candidate_rule, preferred_candidate_idx):
                memory_key = (current, candidate_rule, try_idx, _library_version_hash())
                if use_failure_memory and memory_key in _FAILURE_MEMORY:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](memory-skip)")
                    continue

                attempt = propose_fix(current, candidate_rule, try_idx)
                if attempt is not None and attempt.get('is_valid'):
                    rule_fixed = attempt
                    candidate_reason = f"{this_candidate_reason} (candidate_idx={try_idx})"
                    break
                else:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}]")
                    if use_failure_memory:
                        _FAILURE_MEMORY[memory_key] = True

            if rule_fixed is not None:
                fixed = rule_fixed
                target_rule = candidate_rule
                break

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 "
                              f"({failed_attempts}) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 "
                              f"실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 "
                              f"등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙/candidate {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}

Overwriting src/tools/molecule_editor.py


In [19]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter
status_counter = Counter()
results = []
sample = single_problem_check[:20]

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck인 케이스 ---")
for smi, status, loop_result in results:
    if status == 'stuck':
        print(f"원본: {smi}")
        print(f"결과: {loop_result}\n")

--- 상태 분포 (20개 표본) ---
success: 13
stuck: 7

--- 여전히 stuck인 케이스 ---
원본: CCOCCOCCO
결과: {'status': 'stuck', 'reason': "시도한 규칙/candidate ['Aliphatic_long_chain[idx=0]', 'Aliphatic_long_chain[idx=1]'] 모두 치환 실패", 'reason_detail': "이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 (['Aliphatic_long_chain[idx=0]', 'Aliphatic_long_chain[idx=1]']) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.", 'final_smiles': 'CCOCCOCCO', 'history': [{'step': 0, 'smiles': 'CCOCCOCCO', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [1, 2, 3, 4]}]}], 'skipped_rules': [], 'skipped_details': []}

원본: COc1ccccc1N1CCN(CCCCNC(=O)c2ccc(-c3ccc(C(C)=O)cc3)cc2)CC1
결과: {'status': 'stuck', 'reason': "시도한 규칙/candidate ['Aliphatic_long_chain[idx=0]', 'Aliphatic_long_chain[idx=1]'] 모두 치환 실패", 'reason_detail': "이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 (['Aliphatic_long_chain[idx=0]', 'Aliphatic_long_chain[idx=1]']) 모두 실행에 실패했습니다(memo

In [32]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_multi_bond":
        pairs = candidate.get("target_pairs_in_pattern", info.get("target_pairs_in_pattern"))
        ring_atoms_pattern = candidate.get("ring_atoms_in_pattern", info.get("ring_atoms_in_pattern"))
        ring_bonds_pattern = candidate.get("ring_bonds_in_pattern", info.get("ring_bonds_in_pattern"))

        for pair in pairs:
            idx_c = match[pair[0]]
            idx_o = match[pair[1]]
            bond = rwmol.GetBondBetweenAtoms(idx_c, idx_o)
            if bond is None:
                return None
            bond.SetBondType(Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
            rwmol.GetAtomWithIdx(idx_c).SetNoImplicit(False)

        ring_indices = [match[i] for i in ring_atoms_pattern]
        for a in ring_indices:
            rwmol.GetAtomWithIdx(a).SetIsAromatic(True)

        for b1, b2 in ring_bonds_pattern:
            bidx1, bidx2 = match[b1], match[b2]
            rbond = rwmol.GetBondBetweenAtoms(bidx1, bidx2)
            if rbond is None:
                continue
            rbond.SetBondType(Chem.BondType.AROMATIC)
            rbond.SetIsAromatic(True)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "remove_atom":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust4(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust4(center_idx, to_remove)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(center_new).SetFormalCharge(0)

    elif edit_type == "cleave_bond":
        pair = candidate["cleave_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)
        for idx in (idx1, idx2):
            rwmol.GetAtomWithIdx(idx).SetNoImplicit(False)

    elif edit_type == "open_epoxide":
        pair = candidate["break_pair_in_pattern"]
        idx_o = match[pair[0]]
        idx_c_break = match[pair[1]]

        bond = rwmol.GetBondBetweenAtoms(idx_o, idx_c_break)
        if bond is None:
            return None
        rwmol.RemoveBond(idx_o, idx_c_break)

        frag = Chem.MolFromSmiles("O")
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(idx_c_break, offset, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx_c_break).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(offset).SetNoImplicit(False)

    elif edit_type == "insert_atom":
        pair = candidate["insert_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]

        if rwmol.GetAtomWithIdx(idx1).GetSymbol() == 'O' or rwmol.GetAtomWithIdx(idx2).GetSymbol() == 'O':
            return None

        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)

        new_atom = Chem.Atom(candidate["param"])
        new_idx = rwmol.AddAtom(new_atom)
        rwmol.AddBond(idx1, new_idx, Chem.BondType.SINGLE)
        rwmol.AddBond(new_idx, idx2, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx1).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx2).SetNoImplicit(False)

    elif edit_type == "insert_atom_multi_chain":
        # [재설계, notebook 45-2] 실제 RDKit BRENK Aliphatic_long_chain
        # SMARTS는 "[R0&D2][R0&D2][R0&D2][R0&D2]" -> 원소 종류와 무관하게
        # 고리 밖의 "연속 4개 degree-2 원자"를 잡는다. 즉 에테르 산소를
        # 사슬에 끼워넣어도 그 O 자신도 degree-2라서 연속 구간이 전혀
        # 끊기지 않는다(오히려 길어질 수 있음) -> 지금까지의 "삽입" 전략은
        # 원리적으로 이 규칙을 풀 수 없었음.
        # 해결: degree를 3으로 만드는 "분기(branch)"를 사슬에 규칙적인
        # 간격(<=3칸)으로 달아서 연속 4개 D2 원자 구간을 원천 차단.
        start_idx = match[candidate["chain_start_idx_in_pattern"]]

        def _is_chain_member(atom):
            # 실제 BRENK 정의와 동일: 원소 무관, 고리 밖, degree==2
            return (not atom.GetIsAromatic()) and (not atom.IsInRing()) and atom.GetDegree() == 2

        def _trace_chain(mol, start, avoid):
            # degree==2인 원자는 이웃이 정확히 2개뿐이라 분기 없이
            # 경로가 유일하게 결정됨(들어온 방향 제외 나머지 1개로만 진행)
            chain = []
            current, prev = start, avoid
            while _is_chain_member(mol.GetAtomWithIdx(current)):
                chain.append(current)
                nbs = [n.GetIdx() for n in mol.GetAtomWithIdx(current).GetNeighbors() if n.GetIdx() != prev]
                if len(nbs) != 1:
                    break
                nxt = nbs[0]
                if nxt in chain:
                    break
                prev, current = current, nxt
                if len(chain) > 60:
                    break
            return chain

        start_atom = mol.GetAtomWithIdx(start_idx)
        neighbor_all = [n.GetIdx() for n in start_atom.GetNeighbors()]
        traces = [_trace_chain(mol, nb, start_idx) for nb in neighbor_all]
        traces.sort(key=len, reverse=True)
        left = traces[0] if len(traces) > 0 else []
        right = traces[1] if len(traces) > 1 else []
        chain_atoms = list(reversed(left)) + [start_idx] + right

        n = len(chain_atoms)
        if n < 4:
            return None

        # 연속 4개 이상 D2 구간이 남지 않도록 3칸 간격으로 분기 위치 선정
        insert_positions = list(range(3, n, 4))
        if insert_positions and (n - 1 - insert_positions[-1]) >= 4:
            insert_positions.append(min(n - 1, insert_positions[-1] + 4))
        elif not insert_positions:
            insert_positions = [min(3, n - 1)]

        branch_atomic_num = 6  # 메틸 분기(탄소). 산소로 분기하면 기존 에테르 O 옆에서
                                # O-C-O 패턴이 생겨 'het-C-het_not_in_ring'을 새로 유발함

        added = 0
        for p in insert_positions:
            cand_positions = [p]
            if p - 1 >= 0:
                cand_positions.append(p - 1)
            if p + 1 < n:
                cand_positions.append(p + 1)

            target_idx = None
            for cp in cand_positions:
                a_idx = chain_atoms[cp]
                # 탄소에만 분기(헤테로원자 분기는 원자가 문제를 일으킬 수 있어 회피)
                if rwmol.GetAtomWithIdx(a_idx).GetSymbol() == 'C':
                    target_idx = a_idx
                    break
            if target_idx is None:
                continue

            new_atom = rwmol.AddAtom(Chem.Atom(branch_atomic_num))
            rwmol.AddBond(target_idx, new_atom, Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(target_idx).SetNoImplicit(False)
            added += 1

        if added == 0:
            return None

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        ring_set = set(ring_indices)
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            for n in ratom.GetNeighbors():
                nidx = n.GetIdx()
                if nidx not in ring_set and nidx not in (anchor_idx1, anchor_idx2):
                    return None

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        allow_counterion = candidate.get("allow_counterion", False)
        allow_aromatic_zero_h = candidate.get("allow_aromatic_zero_h", False)

        if edit_type != "cleave_bond" and not allow_counterion and '.' in new_smiles:
            is_valid = False
        for atom in check_mol.GetAtoms():
            if allow_aromatic_zero_h and atom.GetIsAromatic() and atom.GetSymbol() == 'N':
                continue
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [21]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter
status_counter = Counter()
results = []
sample = single_problem_check[:20]

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck인 케이스 ---")
for smi, status, loop_result in results:
    if status == 'stuck':
        print(f"원본: {smi}")
        print(f"결과: {loop_result}\n")

--- 상태 분포 (20개 표본) ---
success: 16
stuck: 2
no_known_fix: 1
max_iterations_reached: 1

--- 여전히 stuck인 케이스 ---
원본: Cc1c(N)nc([C@H](CC(N)=O)NC[C@H](N)C(N)=O)nc1C(=O)N[C@H](C(=O)N[C@H](C)[C@@H](O)[C@H](C)C(=O)N[C@H](C(=O)NCCc1nc(-c2nc(C(=O)NCCCCNC(=N)N)cs2)cs1)[C@@H](C)O)[C@@H](O[C@@H]1O[C@@H](CO)[C@@H](O)[C@H](O)[C@@H]1O[C@H]1O[C@H](CO)[C@@H](O)[C@H](OC(N)=O)[C@@H]1O)c1cnc[nH]1
결과: {'status': 'stuck', 'reason': "시도한 규칙/candidate ['Aliphatic_long_chain[idx=0]', 'Aliphatic_long_chain[idx=1]'] 모두 치환 실패", 'reason_detail': "이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 (['Aliphatic_long_chain[idx=0]', 'Aliphatic_long_chain[idx=1]']) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.", 'final_smiles': 'Cc1c(N)nc([C@H](CC(N)=O)NC[C@H](N)C(N)=O)nc1C(=O)N[C@H](C(=O)N[C@H](C)[C@@H](O)[C@H](C)C(=O)N[C@H](C(=O)NCCc1nc(-c2nc(C(=O)NCCCCNC(=N)N)cs2)cs1)[C@@H](C)O)[C@@H](O[C@@H]1O[C@@H](CO)[C@@H](O)[C@H](O)[C@@H]1O[C@H]1O[C@H](CO)[C

In [25]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_multi_bond":
        pairs = candidate.get("target_pairs_in_pattern", info.get("target_pairs_in_pattern"))
        ring_atoms_pattern = candidate.get("ring_atoms_in_pattern", info.get("ring_atoms_in_pattern"))
        ring_bonds_pattern = candidate.get("ring_bonds_in_pattern", info.get("ring_bonds_in_pattern"))

        for pair in pairs:
            idx_c = match[pair[0]]
            idx_o = match[pair[1]]
            bond = rwmol.GetBondBetweenAtoms(idx_c, idx_o)
            if bond is None:
                return None
            bond.SetBondType(Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
            rwmol.GetAtomWithIdx(idx_c).SetNoImplicit(False)

        ring_indices = [match[i] for i in ring_atoms_pattern]
        for a in ring_indices:
            rwmol.GetAtomWithIdx(a).SetIsAromatic(True)

        for b1, b2 in ring_bonds_pattern:
            bidx1, bidx2 = match[b1], match[b2]
            rbond = rwmol.GetBondBetweenAtoms(bidx1, bidx2)
            if rbond is None:
                continue
            rbond.SetBondType(Chem.BondType.AROMATIC)
            rbond.SetIsAromatic(True)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "remove_atom":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust4(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust4(center_idx, to_remove)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(center_new).SetFormalCharge(0)

    elif edit_type == "cleave_bond":
        pair = candidate["cleave_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)
        for idx in (idx1, idx2):
            rwmol.GetAtomWithIdx(idx).SetNoImplicit(False)

    elif edit_type == "open_epoxide":
        pair = candidate["break_pair_in_pattern"]
        idx_o = match[pair[0]]
        idx_c_break = match[pair[1]]

        bond = rwmol.GetBondBetweenAtoms(idx_o, idx_c_break)
        if bond is None:
            return None
        rwmol.RemoveBond(idx_o, idx_c_break)

        frag = Chem.MolFromSmiles("O")
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(idx_c_break, offset, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx_c_break).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(offset).SetNoImplicit(False)

    elif edit_type == "insert_atom":
        pair = candidate["insert_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]

        if rwmol.GetAtomWithIdx(idx1).GetSymbol() == 'O' or rwmol.GetAtomWithIdx(idx2).GetSymbol() == 'O':
            return None

        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)

        new_atom = Chem.Atom(candidate["param"])
        new_idx = rwmol.AddAtom(new_atom)
        rwmol.AddBond(idx1, new_idx, Chem.BondType.SINGLE)
        rwmol.AddBond(new_idx, idx2, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx1).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx2).SetNoImplicit(False)

    elif edit_type == "insert_atom_multi_chain":
        # 실제 RDKit BRENK Aliphatic_long_chain SMARTS는
        # "[R0&D2][R0&D2][R0&D2][R0&D2]" -> 원소 무관, degree==2 연속
        # 4개. degree를 3으로 만드는 분기(branch)로 끊는다.
        start_idx = match[candidate["chain_start_idx_in_pattern"]]

        def _is_chain_member(atom):
            return (not atom.GetIsAromatic()) and (not atom.IsInRing()) and atom.GetDegree() == 2

        def _trace_chain(mol, start, avoid):
            chain = []
            current, prev = start, avoid
            while _is_chain_member(mol.GetAtomWithIdx(current)):
                chain.append(current)
                nbs = [n.GetIdx() for n in mol.GetAtomWithIdx(current).GetNeighbors() if n.GetIdx() != prev]
                if len(nbs) != 1:
                    break
                nxt = nbs[0]
                if nxt in chain:
                    break
                prev, current = current, nxt
                if len(chain) > 60:
                    break
            return chain

        start_atom = mol.GetAtomWithIdx(start_idx)
        neighbor_all = [n.GetIdx() for n in start_atom.GetNeighbors()]
        traces = [_trace_chain(mol, nb, start_idx) for nb in neighbor_all]
        traces.sort(key=len, reverse=True)
        left = traces[0] if len(traces) > 0 else []
        right = traces[1] if len(traces) > 1 else []
        chain_atoms = list(reversed(left)) + [start_idx] + right

        n = len(chain_atoms)
        if n < 4:
            return None

        insert_positions = list(range(3, n, 4))
        if insert_positions and (n - 1 - insert_positions[-1]) >= 4:
            insert_positions.append(min(n - 1, insert_positions[-1] + 4))
        elif not insert_positions:
            insert_positions = [min(3, n - 1)]

        branch_atomic_num = candidate.get("param", 8)

        def _find_branch_carbon(p):
            # 우선 바로 옆(p-1,p,p+1)에서 탄소 탐색, 없으면 사슬
            # 전체를 바깥쪽으로 확장하며 탐색(짧은 사슬/헤테로원자
            # 밀집 구간에서도 분기 지점을 반드시 찾기 위함)
            for cp in (p, p - 1, p + 1):
                if 0 <= cp < n:
                    a = rwmol.GetAtomWithIdx(chain_atoms[cp])
                    if a.GetSymbol() == 'C' and a.GetTotalNumHs() > 0:
                        return chain_atoms[cp]
            for radius in range(2, n):
                for cp in (p - radius, p + radius):
                    if 0 <= cp < n:
                        a = rwmol.GetAtomWithIdx(chain_atoms[cp])
                        if a.GetSymbol() == 'C' and a.GetTotalNumHs() > 0:
                            return chain_atoms[cp]

        added = 0
        used_targets = set()
        for p in insert_positions:
            target_idx = _find_branch_carbon(p)
            if target_idx is None or target_idx in used_targets:
                continue
            new_atom = rwmol.AddAtom(Chem.Atom(branch_atomic_num))
            rwmol.AddBond(target_idx, new_atom, Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(target_idx).SetNoImplicit(False)
            used_targets.add(target_idx)
            added += 1

        if added == 0:
            return None

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        ring_set = set(ring_indices)
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            for n in ratom.GetNeighbors():
                nidx = n.GetIdx()
                if nidx not in ring_set and nidx not in (anchor_idx1, anchor_idx2):
                    return None

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        allow_counterion = candidate.get("allow_counterion", False)
        allow_aromatic_zero_h = candidate.get("allow_aromatic_zero_h", False)

        if edit_type != "cleave_bond" and not allow_counterion and '.' in new_smiles:
            is_valid = False
        for atom in check_mol.GetAtoms():
            if allow_aromatic_zero_h and atom.GetIsAromatic() and atom.GetSymbol() == 'N':
                continue
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [22]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib

def _library_version_hash():
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]
            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue
            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue
            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')
            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def _candidate_order_for_rule(rule_name: str, preferred_idx: int):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return [preferred_idx]
    n = len(info['candidates'])
    order = [preferred_idx] if 0 <= preferred_idx < n else []
    order += [i for i in range(n) if i != preferred_idx]
    return order


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini",
                        use_failure_memory: bool = True):
    """진단->치환->재평가를 반복.
    핵심: candidate가 '화학적으로 유효(is_valid)'해도 대상 규칙이 실제로
    해소됐는지 재진단(detect_toxicophores)까지 확인한다. 그렇지 않으면
    항상 valid하지만 문제를 안 고치는 candidate(예: 단순 삽입형)가
    무한 반복 채택되어 진짜 해법(예: 분기형)으로 넘어가지 못하는 문제가
    있었음. 완전 해소가 안 되면 마지막으로 시도한(=대개 더 나은)
    valid 결과를 fallback으로 채택해 다음 iteration에서 계속 개선."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            problem_reason = "규칙 기반(리스트 순서대로)"
            ordered_rules = [p['rule_name'] for p in known_problems]

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                preferred_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if preferred_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                preferred_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도)"

            rule_fixed = None
            fallback_attempt = None
            fallback_used_idx = None
            fallback_reason = None

            for try_idx in _candidate_order_for_rule(candidate_rule, preferred_candidate_idx):
                memory_key = (current, candidate_rule, try_idx, _library_version_hash())
                if use_failure_memory and memory_key in _FAILURE_MEMORY:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](memory-skip)")
                    continue

                attempt = propose_fix(current, candidate_rule, try_idx)
                if attempt is None or not attempt.get('is_valid'):
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}]")
                    if use_failure_memory:
                        _FAILURE_MEMORY[memory_key] = True
                    continue

                # valid해도 실제로 이 규칙이 재진단에서 사라졌는지 확인
                recheck = detect_toxicophores(attempt['new_smiles'])
                still_flagged = any(p['rule_name'] == candidate_rule for p in recheck)

                if not still_flagged:
                    rule_fixed = attempt
                    candidate_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 완전 해소)"
                    break
                else:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](valid이나 미해소)")
                    fallback_attempt = attempt
                    fallback_used_idx = try_idx
                    fallback_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 부분 개선/다음 iteration에서 계속)"

            if rule_fixed is None and fallback_attempt is not None:
                rule_fixed = fallback_attempt
                candidate_reason = fallback_reason

            if rule_fixed is not None:
                fixed = rule_fixed
                target_rule = candidate_rule
                break

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 "
                              f"({failed_attempts}) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 "
                              f"실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 "
                              f"등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙/candidate {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}

Overwriting src/tools/molecule_editor.py


In [24]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter
status_counter = Counter()
results = []
sample = single_problem_check[:20]

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck/no_known_fix/max_iterations인 케이스 ---")
for smi, status, loop_result in results:
    if status != 'success':
        print(f"[{status}] 원본: {smi}")

[07:32:56] Explicit valence for atom # 1 C, 5, is greater than permitted
[07:32:56] Explicit valence for atom # 1 C, 5, is greater than permitted
[07:32:56] Explicit valence for atom # 4 C, 5, is greater than permitted
[07:32:56] Explicit valence for atom # 4 C, 5, is greater than permitted
[07:32:56] Explicit valence for atom # 1 C, 5, is greater than permitted


--- 상태 분포 (20개 표본) ---
success: 14
stuck: 4
no_known_fix: 1
max_iterations_reached: 1

--- 여전히 stuck/no_known_fix/max_iterations인 케이스 ---
[no_known_fix] 원본: CCCCCCCCCCCCSC#N
[stuck] 원본: Cc1c(N)nc([C@H](CC(N)=O)NC[C@H](N)C(N)=O)nc1C(=O)N[C@H](C(=O)N[C@H](C)[C@@H](O)[C@H](C)C(=O)N[C@H](C(=O)NCCc1nc(-c2nc(C(=O)NCCCCNC(=N)N)cs2)cs1)[C@@H](C)O)[C@@H](O[C@@H]1O[C@@H](CO)[C@@H](O)[C@H](O)[C@@H]1O[C@H]1O[C@H](CO)[C@@H](O)[C@H](OC(N)=O)[C@@H]1O)c1cnc[nH]1
[stuck] 원본: CC(C)CCCCCCOC(=O)C1CCCCC1C(=O)OCCCCCCC(C)C
[stuck] 원본: CCCCCCCCCCCCCCCCC(=O)O
[max_iterations_reached] 원본: COCCCCCCCCOCCC[Si](C)(O[Si](C)(C)C)O[Si](C)(C)C
[stuck] 원본: CCCCCCCCOCCC#N


In [26]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_multi_bond":
        pairs = candidate.get("target_pairs_in_pattern", info.get("target_pairs_in_pattern"))
        ring_atoms_pattern = candidate.get("ring_atoms_in_pattern", info.get("ring_atoms_in_pattern"))
        ring_bonds_pattern = candidate.get("ring_bonds_in_pattern", info.get("ring_bonds_in_pattern"))

        for pair in pairs:
            idx_c = match[pair[0]]
            idx_o = match[pair[1]]
            bond = rwmol.GetBondBetweenAtoms(idx_c, idx_o)
            if bond is None:
                return None
            bond.SetBondType(Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
            rwmol.GetAtomWithIdx(idx_c).SetNoImplicit(False)

        ring_indices = [match[i] for i in ring_atoms_pattern]
        for a in ring_indices:
            rwmol.GetAtomWithIdx(a).SetIsAromatic(True)

        for b1, b2 in ring_bonds_pattern:
            bidx1, bidx2 = match[b1], match[b2]
            rbond = rwmol.GetBondBetweenAtoms(bidx1, bidx2)
            if rbond is None:
                continue
            rbond.SetBondType(Chem.BondType.AROMATIC)
            rbond.SetIsAromatic(True)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "remove_atom":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust4(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust4(center_idx, to_remove)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(center_new).SetFormalCharge(0)

    elif edit_type == "cleave_bond":
        pair = candidate["cleave_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)
        for idx in (idx1, idx2):
            rwmol.GetAtomWithIdx(idx).SetNoImplicit(False)

    elif edit_type == "open_epoxide":
        pair = candidate["break_pair_in_pattern"]
        idx_o = match[pair[0]]
        idx_c_break = match[pair[1]]

        bond = rwmol.GetBondBetweenAtoms(idx_o, idx_c_break)
        if bond is None:
            return None
        rwmol.RemoveBond(idx_o, idx_c_break)

        frag = Chem.MolFromSmiles("O")
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(idx_c_break, offset, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx_c_break).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(offset).SetNoImplicit(False)

    elif edit_type == "insert_atom":
        pair = candidate["insert_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]

        if rwmol.GetAtomWithIdx(idx1).GetSymbol() == 'O' or rwmol.GetAtomWithIdx(idx2).GetSymbol() == 'O':
            return None

        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)

        new_atom = Chem.Atom(candidate["param"])
        new_idx = rwmol.AddAtom(new_atom)
        rwmol.AddBond(idx1, new_idx, Chem.BondType.SINGLE)
        rwmol.AddBond(new_idx, idx2, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx1).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx2).SetNoImplicit(False)

    elif edit_type == "insert_atom_multi_chain":
        # 실제 RDKit BRENK Aliphatic_long_chain SMARTS는
        # "[R0&D2][R0&D2][R0&D2][R0&D2]" -> 원소 무관, degree==2 연속
        # 4개. degree를 3으로 만드는 분기(branch)로 끊는다.
        start_idx = match[candidate["chain_start_idx_in_pattern"]]

        def _is_chain_member(atom):
            return (not atom.GetIsAromatic()) and (not atom.IsInRing()) and atom.GetDegree() == 2

        def _trace_chain(mol, start, avoid):
            chain = []
            current, prev = start, avoid
            while _is_chain_member(mol.GetAtomWithIdx(current)):
                chain.append(current)
                nbs = [n.GetIdx() for n in mol.GetAtomWithIdx(current).GetNeighbors() if n.GetIdx() != prev]
                if len(nbs) != 1:
                    break
                nxt = nbs[0]
                if nxt in chain:
                    break
                prev, current = current, nxt
                if len(chain) > 60:
                    break
            return chain

        start_atom = mol.GetAtomWithIdx(start_idx)
        neighbor_all = [n.GetIdx() for n in start_atom.GetNeighbors()]
        traces = [_trace_chain(mol, nb, start_idx) for nb in neighbor_all]
        traces.sort(key=len, reverse=True)
        left = traces[0] if len(traces) > 0 else []
        right = traces[1] if len(traces) > 1 else []
        chain_atoms = list(reversed(left)) + [start_idx] + right

        n = len(chain_atoms)
        if n < 4:
            return None

        insert_positions = list(range(3, n, 4))
        if insert_positions and (n - 1 - insert_positions[-1]) >= 4:
            insert_positions.append(min(n - 1, insert_positions[-1] + 4))
        elif not insert_positions:
            insert_positions = [min(3, n - 1)]

        branch_atomic_num = candidate.get("param", 8)

        def _find_branch_carbon(p):
            # 우선 바로 옆(p-1,p,p+1)에서 탄소 탐색, 없으면 사슬
            # 전체를 바깥쪽으로 확장하며 탐색(짧은 사슬/헤테로원자
            # 밀집 구간에서도 분기 지점을 반드시 찾기 위함)
            for cp in (p, p - 1, p + 1):
                if 0 <= cp < n and rwmol.GetAtomWithIdx(chain_atoms[cp]).GetSymbol() == 'C':
                    return chain_atoms[cp]
            for radius in range(2, n):
                for cp in (p - radius, p + radius):
                    if 0 <= cp < n and rwmol.GetAtomWithIdx(chain_atoms[cp]).GetSymbol() == 'C':
                        return chain_atoms[cp]
            return None

        added = 0
        used_targets = set()
        for p in insert_positions:
            target_idx = _find_branch_carbon(p)
            if target_idx is None or target_idx in used_targets:
                continue
            new_atom = rwmol.AddAtom(Chem.Atom(branch_atomic_num))
            rwmol.AddBond(target_idx, new_atom, Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(target_idx).SetNoImplicit(False)
            used_targets.add(target_idx)
            added += 1

        if added == 0:
            return None

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        ring_set = set(ring_indices)
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            for n in ratom.GetNeighbors():
                nidx = n.GetIdx()
                if nidx not in ring_set and nidx not in (anchor_idx1, anchor_idx2):
                    return None

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        allow_counterion = candidate.get("allow_counterion", False)
        allow_aromatic_zero_h = candidate.get("allow_aromatic_zero_h", False)

        if edit_type != "cleave_bond" and not allow_counterion and '.' in new_smiles:
            is_valid = False
        for atom in check_mol.GetAtoms():
            if allow_aromatic_zero_h and atom.GetIsAromatic() and atom.GetSymbol() == 'N':
                continue
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [27]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()
from collections import Counter
status_counter = Counter()
results = []
sample = single_problem_check[:20]

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck/no_known_fix/max_iterations인 케이스 ---")
for smi, status, loop_result in results:
    if status != 'success':
        print(f"[{status}] 원본: {smi}")

[07:37:41] Explicit valence for atom # 1 C, 5, is greater than permitted
[07:37:41] Explicit valence for atom # 1 C, 5, is greater than permitted


--- 상태 분포 (20개 표본) ---
success: 14
stuck: 4
no_known_fix: 1
max_iterations_reached: 1

--- 여전히 stuck/no_known_fix/max_iterations인 케이스 ---
[no_known_fix] 원본: CCCCCCCCCCCCSC#N
[stuck] 원본: Cc1c(N)nc([C@H](CC(N)=O)NC[C@H](N)C(N)=O)nc1C(=O)N[C@H](C(=O)N[C@H](C)[C@@H](O)[C@H](C)C(=O)N[C@H](C(=O)NCCc1nc(-c2nc(C(=O)NCCCCNC(=N)N)cs2)cs1)[C@@H](C)O)[C@@H](O[C@@H]1O[C@@H](CO)[C@@H](O)[C@H](O)[C@@H]1O[C@H]1O[C@H](CO)[C@@H](O)[C@H](OC(N)=O)[C@@H]1O)c1cnc[nH]1
[stuck] 원본: CC(C)CCCCCCOC(=O)C1CCCCC1C(=O)OCCCCCCC(C)C
[stuck] 원본: CCCCCCCCCCCCCCCCC(=O)O
[max_iterations_reached] 원본: COCCCCCCCCOCCC[Si](C)(O[Si](C)(C)C)O[Si](C)(C)C
[stuck] 원본: CCCCCCCCOCCC#N


[07:37:41] Explicit valence for atom # 4 C, 5, is greater than permitted
[07:37:41] Explicit valence for atom # 4 C, 5, is greater than permitted
[07:37:41] Explicit valence for atom # 1 C, 5, is greater than permitted


In [28]:
with open('src/tools/atom_editor.py') as f:
    content = f.read()
print('TotalNumHs' in content)

True


In [30]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter
status_counter = Counter()
results = []
sample = single_problem_check[:20]

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck/no_known_fix/max_iterations인 케이스 ---")
for smi, status, loop_result in results:
    if status != 'success':
        print(f"[{status}] 원본: {smi}")

[07:42:30] Explicit valence for atom # 1 C, 5, is greater than permitted
[07:42:30] Explicit valence for atom # 1 C, 5, is greater than permitted


--- 상태 분포 (20개 표본) ---
success: 14
stuck: 4
no_known_fix: 1
max_iterations_reached: 1

--- 여전히 stuck/no_known_fix/max_iterations인 케이스 ---
[no_known_fix] 원본: CCCCCCCCCCCCSC#N
[stuck] 원본: Cc1c(N)nc([C@H](CC(N)=O)NC[C@H](N)C(N)=O)nc1C(=O)N[C@H](C(=O)N[C@H](C)[C@@H](O)[C@H](C)C(=O)N[C@H](C(=O)NCCc1nc(-c2nc(C(=O)NCCCCNC(=N)N)cs2)cs1)[C@@H](C)O)[C@@H](O[C@@H]1O[C@@H](CO)[C@@H](O)[C@H](O)[C@@H]1O[C@H]1O[C@H](CO)[C@@H](O)[C@H](OC(N)=O)[C@@H]1O)c1cnc[nH]1
[stuck] 원본: CC(C)CCCCCCOC(=O)C1CCCCC1C(=O)OCCCCCCC(C)C
[stuck] 원본: CCCCCCCCCCCCCCCCC(=O)O
[max_iterations_reached] 원본: COCCCCCCCCOCCC[Si](C)(O[Si](C)(C)C)O[Si](C)(C)C
[stuck] 원본: CCCCCCCCOCCC#N


[07:42:30] Explicit valence for atom # 4 C, 5, is greater than permitted
[07:42:30] Explicit valence for atom # 4 C, 5, is greater than permitted
[07:42:30] Explicit valence for atom # 1 C, 5, is greater than permitted


In [31]:
smi = 'CCCCCCCCOCCC#N'
r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
print(r['status'])
print(r['reason'] if 'reason' in r else '')
print(r['reason_detail'] if 'reason_detail' in r else '')
for h in r['history']:
    print(h)

stuck
시도한 규칙/candidate ['Aliphatic_long_chain[idx=0](memory-skip)', 'Aliphatic_long_chain[idx=1](memory-skip)'] 모두 치환 실패
이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 (['Aliphatic_long_chain[idx=0](memory-skip)', 'Aliphatic_long_chain[idx=1](memory-skip)']) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.
{'step': 0, 'smiles': 'CCCCCCCCOCCC#N', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [1, 2, 3, 4]}]}
{'step': 1, 'smiles': 'N#CCC(O)OCCCC(O)CCCCO', 'fixed_rule': 'Aliphatic_long_chain', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'multi-ether chain (multiple O inserted for long chains)', 'candidate_reason': '규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도) (candidate_idx=1, 부분 개선/다음 iteration에서 계속)', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [5, 6, 7, 8]}, {'rule_name': 'het-C-het_not_in_ring', 'atom_indices': [3, 4, 5]}]}
{'step': 2, 'smiles': 'N#CCO

In [34]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter
status_counter = Counter()
results = []
sample = single_problem_check[:20]

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck/no_known_fix/max_iterations인 케이스 (상세) ---")
for smi, status, loop_result in results:
    if status == 'success':
        continue
    print(f"\n=== [{status}] 원본: {smi} ===")
    print("reason:", loop_result.get('reason', ''))
    print("reason_detail:", loop_result.get('reason_detail', ''))
    for h in loop_result['history']:
        print(" ", h)

--- 상태 분포 (20개 표본) ---
success: 14
stuck: 5
no_known_fix: 1

--- 여전히 stuck/no_known_fix/max_iterations인 케이스 (상세) ---

=== [no_known_fix] 원본: CCCCCCCCCCCCSC#N ===
reason: 
reason_detail: 
  {'step': 0, 'smiles': 'CCCCCCCCCCCCSC#N', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [1, 2, 3, 4]}, {'rule_name': 'cyanate_/aminonitrile_/thiocyanate', 'atom_indices': [12, 13, 14]}]}
  {'step': 1, 'smiles': 'CCC(C)CCCC(C)CCCC(C)CSC#N', 'fixed_rule': 'Aliphatic_long_chain', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'multi-ether chain (multiple O inserted for long chains)', 'candidate_reason': '규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도) (candidate_idx=1, 완전 해소)', 'problems': [{'rule_name': 'cyanate_/aminonitrile_/thiocyanate', 'atom_indices': [15, 16, 17]}]}

=== [stuck] 원본: Cc1c(N)nc([C@H](CC(N)=O)NC[C@H](N)C(N)=O)nc1C(=O)N[C@H](C(=O)N[C@H](C)[C@@H](O)[C@H](C)C(=O)N[C@H](C(=O)NCCc1nc(-c2nc(C(=O)NCCCCNC(=N)N)cs2)cs1)[C@@H](C)O)[C@@H](O[C@@H]1O[C@@H](CO)[C

In [36]:
info = get_replacement_candidates('Aliphatic_long_chain')
print("problem_smarts:", info['problem_smarts'])
for i, c in enumerate(info['candidates']):
    print(f"\ncandidate idx={i}")
    for k, v in c.items():
        print(f"  {k}: {v}")

r = iterative_fix_loop('COCCOCCOCCOC', max_iterations=10, candidate_idx=0)
print(r['status'])
for h in r['history']:
    print(h)

problem_smarts: [C,O;!$([OX2]C=O);!$([OX2]c)][C,O;!$([OX2]C=O);!$([OX2]c)][C,O;!$([OX2]C=O);!$([OX2]c)][C,O;!$([OX2]C=O);!$([OX2]c)]

candidate idx=0
  edit_type: insert_atom
  insert_pair_in_pattern: (1, 2)
  param: 8
  name: ether-inserted chain (O in middle)
  rationale: 탄소 4개 이상 연속된 지방족(비고리) 사슬은 과도한 지용성을 유발해 막 축적, 대사 불안정성, 부적절한 약물동태(반감기 과다 연장 등)를 일으킬 수 있음. 사슬 중간에 산소(에테르)를 삽입해 극성을 높이고 지용성을 낮추는 것은 실제 의약화학에서 널리 쓰이는 bioisostere 전략임 (검증 필요)

candidate idx=1
  edit_type: insert_atom_multi_chain
  chain_start_idx_in_pattern: 0
  name: multi-ether chain (multiple O inserted for long chains)
  rationale: 매우 긴 지방족 사슬(수 회 반복이 필요한 경우)에 대해, 4탄소 간격마다 산소를 동시에 여러 개 삽입하여 한 번에 극성을 분산시킴 (검증 필요, 긴 사슬 전용)
stuck
{'step': 0, 'smiles': 'COCCOCCOCCOC', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [1, 2, 3, 4]}]}
{'step': 1, 'smiles': 'COCC(C)OCCOC(C)COC', 'fixed_rule': 'Aliphatic_long_chain', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'multi-ether chain (multiple O inserte

In [37]:
!cat src/tools/atom_editor.py

from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
  

In [41]:
%%writefile src/tools/atom_editor.py

from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_multi_bond":
        pairs = candidate.get("target_pairs_in_pattern", info.get("target_pairs_in_pattern"))
        ring_atoms_pattern = candidate.get("ring_atoms_in_pattern", info.get("ring_atoms_in_pattern"))
        ring_bonds_pattern = candidate.get("ring_bonds_in_pattern", info.get("ring_bonds_in_pattern"))

        for pair in pairs:
            idx_c = match[pair[0]]
            idx_o = match[pair[1]]
            bond = rwmol.GetBondBetweenAtoms(idx_c, idx_o)
            if bond is None:
                return None
            bond.SetBondType(Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
            rwmol.GetAtomWithIdx(idx_c).SetNumExplicitHs(0)
            rwmol.GetAtomWithIdx(idx_c).SetNoImplicit(False)

        ring_indices = [match[i] for i in ring_atoms_pattern]
        for a in ring_indices:
            rwmol.GetAtomWithIdx(a).SetIsAromatic(True)

        for b1, b2 in ring_bonds_pattern:
            bidx1, bidx2 = match[b1], match[b2]
            rbond = rwmol.GetBondBetweenAtoms(bidx1, bidx2)
            if rbond is None:
                continue
            rbond.SetBondType(Chem.BondType.AROMATIC)
            rbond.SetIsAromatic(True)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "remove_substituent":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        upgrade_idx = match[candidate["upgrade_bond_to_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}

        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        visited.add(upgrade_idx)
        upgrade_atom = mol.GetAtomWithIdx(upgrade_idx)
        for n in upgrade_atom.GetNeighbors():
            if n.GetIdx() != center_idx and n.GetIdx() not in to_remove:
                stack2 = [n.GetIdx()]
                while stack2:
                    cur2 = stack2.pop()
                    if cur2 in visited:
                        continue
                    visited.add(cur2)
                    to_remove.add(cur2)
                    for n2 in mol.GetAtomWithIdx(cur2).GetNeighbors():
                        if n2.GetIdx() not in visited:
                            stack2.append(n2.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust3(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust3(center_idx, to_remove)
        upgrade_new = _adjust3(upgrade_idx, to_remove)

        bond = rwmol.GetBondBetweenAtoms(center_new, upgrade_new)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.DOUBLE)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(upgrade_new).SetNoImplicit(False)

    elif edit_type == "remove_atom":
        remove_idx = match[candidate["remove_idx_in_pattern"]]
        center_idx = match[candidate.get("center_idx_in_pattern", 0)]

        to_remove = set()
        visited = {center_idx}
        stack = [remove_idx]
        while stack:
            cur = stack.pop()
            if cur in visited:
                continue
            visited.add(cur)
            to_remove.add(cur)
            for n in mol.GetAtomWithIdx(cur).GetNeighbors():
                if n.GetIdx() not in visited:
                    stack.append(n.GetIdx())

        for ridx in sorted(to_remove, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust4(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        center_new = _adjust4(center_idx, to_remove)
        rwmol.GetAtomWithIdx(center_new).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(center_new).SetFormalCharge(0)

    elif edit_type == "cleave_bond":
        pair = candidate["cleave_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)
        for idx in (idx1, idx2):
            rwmol.GetAtomWithIdx(idx).SetNoImplicit(False)

    elif edit_type == "open_epoxide":
        pair = candidate["break_pair_in_pattern"]
        idx_o = match[pair[0]]
        idx_c_break = match[pair[1]]

        bond = rwmol.GetBondBetweenAtoms(idx_o, idx_c_break)
        if bond is None:
            return None
        rwmol.RemoveBond(idx_o, idx_c_break)

        frag = Chem.MolFromSmiles("O")
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(idx_c_break, offset, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx_o).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx_c_break).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(offset).SetNoImplicit(False)

    elif edit_type == "insert_atom":
        pair = candidate["insert_pair_in_pattern"]
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]

        if rwmol.GetAtomWithIdx(idx1).GetSymbol() == 'O' or rwmol.GetAtomWithIdx(idx2).GetSymbol() == 'O':
            return None

        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        rwmol.RemoveBond(idx1, idx2)

        new_atom = Chem.Atom(candidate["param"])
        new_idx = rwmol.AddAtom(new_atom)
        rwmol.AddBond(idx1, new_idx, Chem.BondType.SINGLE)
        rwmol.AddBond(new_idx, idx2, Chem.BondType.SINGLE)

        rwmol.GetAtomWithIdx(idx1).SetNoImplicit(False)
        rwmol.GetAtomWithIdx(idx2).SetNoImplicit(False)

    elif edit_type == "insert_atom_multi_chain":
        # 실제 RDKit BRENK Aliphatic_long_chain SMARTS는
        # "[R0&D2][R0&D2][R0&D2][R0&D2]" -> 원소 무관, degree==2 연속
        # 4개. degree를 3으로 만드는 분기(branch)로 끊는다.
        # matches[0]만 쓰면 우연히 말단/분기 원자를 앵커로 잡아 사슬
        # 추적이 즉시 끊길 수 있어(n<4), 모든 매치를 후보로 시도해서
        # 실제로 가장 긴 사슬이 나오는 앵커를 채택한다.

        def _is_chain_member(atom):
            return (not atom.GetIsAromatic()) and (not atom.IsInRing()) and atom.GetDegree() == 2

        def _trace_chain(mol, start, avoid):
            chain = []
            current, prev = start, avoid
            while _is_chain_member(mol.GetAtomWithIdx(current)):
                chain.append(current)
                nbs = [n.GetIdx() for n in mol.GetAtomWithIdx(current).GetNeighbors() if n.GetIdx() != prev]
                if len(nbs) != 1:
                    break
                nxt = nbs[0]
                if nxt in chain:
                    break
                prev, current = current, nxt
                if len(chain) > 60:
                    break
            return chain

        chain_pos = candidate["chain_start_idx_in_pattern"]
        chain_atoms = []
        for m in matches:
            cand_start = m[chain_pos]
            cand_atom = mol.GetAtomWithIdx(cand_start)
            cand_neighbors = [n.GetIdx() for n in cand_atom.GetNeighbors()]
            cand_traces = [_trace_chain(mol, nb, cand_start) for nb in cand_neighbors]
            cand_traces.sort(key=len, reverse=True)
            cleft = cand_traces[0] if len(cand_traces) > 0 else []
            cright = cand_traces[1] if len(cand_traces) > 1 else []
            cand_chain = list(reversed(cleft)) + [cand_start] + cright
            if len(cand_chain) > len(chain_atoms):
                chain_atoms = cand_chain

        n = len(chain_atoms)
        if n < 4:
            return None

        insert_positions = list(range(3, n, 4))
        if insert_positions and (n - 1 - insert_positions[-1]) >= 4:
            insert_positions.append(min(n - 1, insert_positions[-1] + 4))
        elif not insert_positions:
            insert_positions = [min(3, n - 1)]

        branch_atomic_num = 6  # 메틸 분기(탄소). 산소로 분기하면 기존 에테르 O 옆에서
                                # O-C-O 패턴이 생겨 'het-C-het_not_in_ring'을 새로 유발함

        def _find_branch_carbon(p):
            for cp in (p, p - 1, p + 1):
                if 0 <= cp < n:
                    a = rwmol.GetAtomWithIdx(chain_atoms[cp])
                    if a.GetSymbol() == 'C' and a.GetTotalNumHs() > 0:
                        return chain_atoms[cp]
            for radius in range(2, n):
                for cp in (p - radius, p + radius):
                    if 0 <= cp < n:
                        a = rwmol.GetAtomWithIdx(chain_atoms[cp])
                        if a.GetSymbol() == 'C' and a.GetTotalNumHs() > 0:
                            return chain_atoms[cp]
            return None

        added = 0
        used_targets = set()
        for p in insert_positions:
            target_idx = _find_branch_carbon(p)
            if target_idx is None or target_idx in used_targets:
                continue
            new_atom = rwmol.AddAtom(Chem.Atom(branch_atomic_num))
            rwmol.AddBond(target_idx, new_atom, Chem.BondType.SINGLE)
            rwmol.GetAtomWithIdx(target_idx).SetNoImplicit(False)
            used_targets.add(target_idx)
            added += 1

        if added == 0:
            return None

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        ring_set = set(ring_indices)
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            for n in ratom.GetNeighbors():
                nidx = n.GetIdx()
                if nidx not in ring_set and nidx not in (anchor_idx1, anchor_idx2):
                    return None

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        allow_counterion = candidate.get("allow_counterion", False)
        allow_aromatic_zero_h = candidate.get("allow_aromatic_zero_h", False)

        if edit_type != "cleave_bond" and not allow_counterion and '.' in new_smiles:
            is_valid = False
        for atom in check_mol.GetAtoms():
            if allow_aromatic_zero_h and atom.GetIsAromatic() and atom.GetSymbol() == 'N':
                continue
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }


Overwriting src/tools/atom_editor.py


In [42]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter
status_counter = Counter()
results = []
sample = single_problem_check[:20]

for smi in sample:
    loop_result = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status = loop_result['status']
    status_counter[status] += 1
    results.append((smi, status, loop_result))

print("--- 상태 분포 (20개 표본) ---")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

print("\n--- 여전히 stuck/no_known_fix/max_iterations인 케이스 (상세) ---")
for smi, status, loop_result in results:
    if status == 'success':
        continue
    print(f"\n=== [{status}] 원본: {smi} ===")
    print("reason:", loop_result.get('reason', ''))
    for h in loop_result['history']:
        print(" ", h)

--- 상태 분포 (20개 표본) ---
success: 17
no_known_fix: 3

--- 여전히 stuck/no_known_fix/max_iterations인 케이스 (상세) ---

=== [no_known_fix] 원본: CCCCCCCCCCCCSC#N ===
reason: 
  {'step': 0, 'smiles': 'CCCCCCCCCCCCSC#N', 'problems': [{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [1, 2, 3, 4]}, {'rule_name': 'cyanate_/aminonitrile_/thiocyanate', 'atom_indices': [12, 13, 14]}]}
  {'step': 1, 'smiles': 'CCC(C)CCCC(C)CCCC(C)CSC#N', 'fixed_rule': 'Aliphatic_long_chain', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'multi-ether chain (multiple O inserted for long chains)', 'candidate_reason': '규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도) (candidate_idx=1, 완전 해소)', 'problems': [{'rule_name': 'cyanate_/aminonitrile_/thiocyanate', 'atom_indices': [15, 16, 17]}]}

=== [no_known_fix] 원본: Cc1c(N)nc([C@H](CC(N)=O)NC[C@H](N)C(N)=O)nc1C(=O)N[C@H](C(=O)N[C@H](C)[C@@H](O)[C@H](C)C(=O)N[C@H](C(=O)NCCc1nc(-c2nc(C(=O)NCCCCNC(=N)N)cs2)cs1)[C@@H](C)O)[C@@H](O[C@@H]1O[C@@H](CO)[C@@H](O)[C@H](O)[C@

In [43]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
clear_failure_memory()

from collections import Counter

# 1) Aliphatic_long_chain 단독 문제 95개 전체 재측정
status_counter_95 = Counter()
stuck_95 = []
for smi in single_problem_check:
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status_counter_95[r['status']] += 1
    if r['status'] not in ('success',):
        # Aliphatic_long_chain 자체가 완전 해소됐는지는 history로 별도 확인
        resolved_aliphatic = any(
            h.get('fixed_rule') == 'Aliphatic_long_chain' and '완전 해소' in (h.get('candidate_reason') or '')
            for h in r['history']
        )
        stuck_95.append((smi, r['status'], resolved_aliphatic))

print("=== Aliphatic_long_chain 단독 문제 95개 ===")
for status, count in status_counter_95.most_common():
    print(f"{status}: {count}")
n_aliphatic_actually_stuck = sum(1 for _, _, resolved in stuck_95 if not resolved)
print(f"\nAliphatic_long_chain 자체가 진짜로 안 풀린 케이스: {n_aliphatic_actually_stuck} / {len(single_problem_check)}")
if n_aliphatic_actually_stuck:
    print("--- 진짜로 안 풀린 것들 ---")
    for smi, status, resolved in stuck_95:
        if not resolved:
            print(f"[{status}] {smi}")

# 2) valid set 전체에 대해 다른 규칙 회귀 확인
print("\n\n=== valid set 전체 회귀 확인 ===")
status_counter_all = Counter()
new_stuck_rules = Counter()  # stuck/no_known_fix에서 어떤 규칙이 최종 장애물인지

for smi in data['smiles_valid']:
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    status_counter_all[r['status']] += 1
    if r['status'] in ('stuck', 'no_known_fix', 'max_iterations_reached', 'cycle_detected'):
        last_problems = r['history'][-1].get('problems', [])
        for p in last_problems:
            new_stuck_rules[p['rule_name']] += 1

print("--- 상태 분포 (valid set 전체) ---")
for status, count in status_counter_all.most_common():
    print(f"{status}: {count}")

print("\n--- 최종 장애물이 된 규칙 빈도 상위 15개 ---")
for rule, count in new_stuck_rules.most_common(15):
    print(f"{rule}: {count}")

=== Aliphatic_long_chain 단독 문제 95개 ===
success: 75
no_known_fix: 10
stuck: 10

Aliphatic_long_chain 자체가 진짜로 안 풀린 케이스: 10 / 95
--- 진짜로 안 풀린 것들 ---
[stuck] OCCNCc1ccccc1
[stuck] CCCc1c(OCCCOc2ccc(OCC(=O)O)cc2)ccc(C(C)=O)c1O
[stuck] CO[Si](CCCNCCN)(OC)OC
[stuck] N#CCNCC#N
[stuck] c1ccc(CNCCNCc2ccccc2)cc1
[stuck] CNCCC=C1c2ccccc2CCc2ccccc21
[stuck] CCCCCCCN(CC)CCCC(O)c1ccc(NS(C)(=O)=O)cc1.CCCCCCCN(CC)CCCC(O)c1ccc(NS(C)(=O)=O)cc1
[stuck] NCCNCCNCCNCCN
[stuck] CCC(C(=O)OCCOCCN(CC)CC)c1ccccc1.O=C(O)CC(O)(CC(=O)O)C(=O)O
[stuck] CN(C)CCCNCCCN


=== valid set 전체 회귀 확인 ===


[08:10:42] Incomplete atom labelling, cannot make bond


--- 상태 분포 (valid set 전체) ---
success: 903
no_known_fix: 136
stuck: 134

--- 최종 장애물이 된 규칙 빈도 상위 15개 ---
Oxygen-nitrogen_single_bond: 38
Sulfonic_acid_2: 30
phosphor: 28
Aliphatic_long_chain: 21
heavy_metal: 16
quaternary_nitrogen_1: 15
beta-keto/anhydride: 13
iodine: 12
halogenated_ring_1: 10
Polycyclic_aromatic_hydrocarbon_2: 6
>_2_ester_groups: 6
imine_1_guanidine: 5
imine_2: 5
cyanate_/aminonitrile_/thiocyanate: 5
het_thio_666_A(13): 5


In [44]:
!cd /content/laidd-2026 && git status
!cd /content/laidd-2026 && git log --oneline -5

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   src/tools/atom_editor.py
	modified:   src/tools/molecule_editor.py

no changes added to commit (use "git add" and/or "git commit -a")
5ede2de (HEAD -> main, origin/main, origin/HEAD) final batch & test set validation
1eb4bf7 Add FINAL_ALL_RESULTS.json: comprehensive consolidated summary of all session results (library growth, valid/test set metrics, baseline models, ablation, 3-agent batch validation, precedent library, docking verification, multi-objective scoring, safety incidents, coverage expansion) for proposal reference.
06a1de6 Test set final verification (single use, post-development freeze): coverage 49.9%, success rate 50%, partial improvement 72% - closely matching valid set numbers (50.2%/52%/80%), confirming no overfitting to the validati

In [45]:
import subprocess
from collections import Counter

def run_valid_set():
    importlib.reload(src.tools.replacement_library)
    importlib.reload(src.tools.atom_editor)
    importlib.reload(src.tools.molecule_editor)
    from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
    clear_failure_memory()

    status_counter = Counter()
    blocking_rules = Counter()
    errors = []

    for smi in data['smiles_valid']:
        try:
            r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
        except Exception as e:
            errors.append((smi, repr(e)))
            continue
        status_counter[r['status']] += 1
        if r['status'] in ('stuck', 'no_known_fix', 'max_iterations_reached', 'cycle_detected'):
            last_problems = r['history'][-1].get('problems', [])
            for p in last_problems:
                blocking_rules[p['rule_name']] += 1

    return status_counter, blocking_rules, errors

# 1) 원본(커밋된 상태)으로 되돌려서 실행
print(">>> git stash로 원본 복원 중...")
subprocess.run(['git', 'stash'], cwd='/content/laidd-2026', check=True)

print(">>> 원본 버전으로 valid set 실행...")
baseline_status, baseline_rules, baseline_errors = run_valid_set()

# 2) 수정 버전 복원
print("\n>>> git stash pop으로 수정 버전 복원 중...")
subprocess.run(['git', 'stash', 'pop'], cwd='/content/laidd-2026', check=True)

print(">>> 수정 버전으로 valid set 실행...")
current_status, current_rules, current_errors = run_valid_set()

# 3) 비교
print("\n\n=== 상태 분포 비교 (원본 -> 수정본) ===")
all_statuses = set(baseline_status) | set(current_status)
for status in sorted(all_statuses):
    b = baseline_status.get(status, 0)
    c = current_status.get(status, 0)
    marker = " <-- 변화" if b != c else ""
    print(f"{status}: {b} -> {c}{marker}")

print(f"\n원본 실행 중 예외 발생: {len(baseline_errors)}건")
print(f"수정본 실행 중 예외 발생: {len(current_errors)}건")
if current_errors:
    print("--- 수정본에서 새로 발생한 예외 (상위 5개) ---")
    for smi, err in current_errors[:5]:
        print(f"{smi}: {err}")

print("\n=== 규칙별 최종 장애물 빈도 비교 (원본 -> 수정본) ===")
all_rules = set(baseline_rules) | set(current_rules)
regressed = []
for rule in sorted(all_rules, key=lambda r: -(baseline_rules.get(r,0)+current_rules.get(r,0))):
    b = baseline_rules.get(rule, 0)
    c = current_rules.get(rule, 0)
    if c > b:
        marker = " <-- 악화(회귀 의심)"
        regressed.append(rule)
    elif c < b:
        marker = " <-- 개선"
    else:
        marker = ""
    print(f"{rule}: {b} -> {c}{marker}")

print(f"\n{'⚠️ 악화된 규칙 있음: ' + str(regressed) if regressed else '✅ 악화된 규칙 없음 (Aliphatic_long_chain 외 다른 규칙 전부 유지되거나 개선됨)'}")

>>> git stash로 원본 복원 중...
>>> 원본 버전으로 valid set 실행...


[08:19:44] Incomplete atom labelling, cannot make bond



>>> git stash pop으로 수정 버전 복원 중...
>>> 수정 버전으로 valid set 실행...


[08:20:02] Incomplete atom labelling, cannot make bond




=== 상태 분포 비교 (원본 -> 수정본) ===
max_iterations_reached: 3 -> 0 <-- 변화
no_known_fix: 124 -> 136 <-- 변화
stuck: 252 -> 134 <-- 변화
success: 794 -> 903 <-- 변화

원본 실행 중 예외 발생: 0건
수정본 실행 중 예외 발생: 0건

=== 규칙별 최종 장애물 빈도 비교 (원본 -> 수정본) ===
Aliphatic_long_chain: 140 -> 21 <-- 개선
Oxygen-nitrogen_single_bond: 38 -> 38
Sulfonic_acid_2: 31 -> 30 <-- 개선
phosphor: 28 -> 28
heavy_metal: 15 -> 16 <-- 악화(회귀 의심)
quaternary_nitrogen_1: 15 -> 15
beta-keto/anhydride: 13 -> 13
iodine: 12 -> 12
halogenated_ring_1: 10 -> 10
het-C-het_not_in_ring: 8 -> 5 <-- 개선
>_2_ester_groups: 6 -> 6
Polycyclic_aromatic_hydrocarbon_2: 6 -> 6
imine_1_guanidine: 5 -> 5
acyclic_C=C-O: 5 -> 5
quaternary_nitrogen_2: 5 -> 5
cyanate_/aminonitrile_/thiocyanate: 5 -> 5
phthalimide: 5 -> 5
quinone_A(370): 5 -> 5
imine_2: 5 -> 5
het_thio_666_A(13): 5 -> 5
halogenated_ring_2: 5 -> 5
catechol: 5 -> 5
polyene: 4 -> 4
triflate: 4 -> 4
Perfluorinated_chain: 4 -> 4
cumarine: 4 -> 4
Carbo_cation/anion: 3 -> 3
hydantoin: 3 -> 3
anil_di_alk_B(251):

In [46]:
!git add -A

In [48]:
!git status
!git commit -m "aliphatic-long-chain complete fix"
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/atom_editor.py
	modified:   src/tools/molecule_editor.py

[main 8fd9cd6] aliphatic-long-chain complete fix
 2 files changed, 124 insertions(+), 91 deletions(-)
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [49]:
!git push origin main

Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.98 KiB | 2.98 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   5ede2de..8fd9cd6  main -> main
